Resample to 0.5х0.5х0.625 voxel size

In [ ]:
# !pip install pydicom scipy SimpleITK tqdm

from pathlib import Path

import numpy as np
import pydicom
import SimpleITK as sitk
from scipy.interpolate import RegularGridInterpolator
from tqdm.auto import tqdm


def load_dicom_series(patient_dir):
    """
    Читает серию DICOM.

    Возвращает:
        volume_hu: np.ndarray, shape (R, C, Z)
        row_spacing: float
        col_spacing: float
        z_positions: np.ndarray, shape (Z,)
        ipp0: np.ndarray, shape (3,)
        row_dir: np.ndarray, shape (3,)   - направление axis 0 (Rows)
        col_dir: np.ndarray, shape (3,)   - направление axis 1 (Cols)
        slice_dir: np.ndarray, shape (3,) - направление axis 2 (Slices)
    """
    patient_dir = Path(patient_dir)
    files = sorted(patient_dir.glob("*.dcm"))
    if not files:
        raise ValueError(f"Нет DICOM в {patient_dir}")

    slices = []
    z_positions = []
    ipps = []

    ref_rows = None
    ref_cols = None
    ref_row_spacing = None
    ref_col_spacing = None
    ref_iop = None

    row_dir = None
    col_dir = None
    slice_dir = None

    for fp in files:
        ds = pydicom.dcmread(str(fp), force=True)

        if "PixelData" not in ds:
            continue

        arr = ds.pixel_array.astype(np.float32)

        slope = float(getattr(ds, "RescaleSlope", 1.0))
        intercept = float(getattr(ds, "RescaleIntercept", 0.0))
        arr = arr * slope + intercept   # HU

        pixel_spacing = np.array(ds.PixelSpacing, dtype=np.float64)
        row_spacing = float(pixel_spacing[0])   # Rows
        col_spacing = float(pixel_spacing[1])   # Cols

        iop = np.array(ds.ImageOrientationPatient, dtype=np.float64)

        # По DICOM:
        # iop[:3]  -> direction of rows in patient space for column stepping
        # iop[3:]  -> direction of columns in patient space for row stepping
        #
        # Для массива pixel_array shape (Rows, Cols):
        # axis 0 = Rows -> шаг по iop[3:]
        # axis 1 = Cols -> шаг по iop[:3]
        col_axis_dir = iop[:3]
        row_axis_dir = iop[3:]
        normal = np.cross(col_axis_dir, row_axis_dir)

        ipp = np.array(ds.ImagePositionPatient, dtype=np.float64)
        z = float(np.dot(ipp, normal))

        if ref_rows is None:
            ref_rows = int(ds.Rows)
            ref_cols = int(ds.Columns)
            ref_row_spacing = row_spacing
            ref_col_spacing = col_spacing
            ref_iop = iop.copy()

            row_dir = row_axis_dir / np.linalg.norm(row_axis_dir)
            col_dir = col_axis_dir / np.linalg.norm(col_axis_dir)
            slice_dir = normal / np.linalg.norm(normal)
        else:
            if int(ds.Rows) != ref_rows or int(ds.Columns) != ref_cols:
                raise ValueError(f"{patient_dir.name}: разные размеры кадров")
            if not np.allclose(iop, ref_iop, atol=1e-4):
                raise ValueError(f"{patient_dir.name}: разная ориентация срезов")
            if not np.isclose(row_spacing, ref_row_spacing, atol=1e-6):
                raise ValueError(f"{patient_dir.name}: разный row spacing")
            if not np.isclose(col_spacing, ref_col_spacing, atol=1e-6):
                raise ValueError(f"{patient_dir.name}: разный col spacing")

        slices.append(arr)
        z_positions.append(z)
        ipps.append(ipp)

    if len(slices) < 2:
        raise ValueError(f"{patient_dir.name}: слишком мало срезов")

    z_positions = np.asarray(z_positions, dtype=np.float64)

    dz = np.diff(z_positions)
    if np.all(dz > 0):
        ipp0 = ipps[0]
    elif np.all(dz < 0):
        slices = slices[::-1]
        z_positions = z_positions[::-1]
        ipp0 = ipps[-1]
        slice_dir = -slice_dir
    else:
        raise ValueError(f"{patient_dir.name}: z_positions не монотонны")

    # убираем возможные дублирующиеся позиции
    keep = [0]
    for i in range(1, len(z_positions)):
        if not np.isclose(z_positions[i], z_positions[keep[-1]], atol=1e-4):
            keep.append(i)

    z_positions = z_positions[keep]
    slices = [slices[i] for i in keep]

    volume_hu = np.stack(slices, axis=-1).astype(np.float32)   # (R, C, Z)

    return (
        volume_hu,
        float(ref_row_spacing),
        float(ref_col_spacing),
        z_positions,
        np.asarray(ipp0, dtype=np.float64),
        np.asarray(row_dir, dtype=np.float64),
        np.asarray(col_dir, dtype=np.float64),
        np.asarray(slice_dir, dtype=np.float64),
    )


def resample_trilinear(volume_hu, row_spacing, col_spacing, z_positions,
                       target_spacing=(0.5, 0.5, 0.625),
                       fill_value=-1024.0):
    """
    volume_hu shape: (R, C, Z)
    target_spacing: (row_mm, col_mm, z_mm)
    """
    target_r, target_c, target_z = map(float, target_spacing)

    R, C, Z = volume_hu.shape

    old_r = np.arange(R, dtype=np.float64) * row_spacing
    old_c = np.arange(C, dtype=np.float64) * col_spacing
    old_z = np.asarray(z_positions, dtype=np.float64)

    new_r = np.arange(0.0, old_r[-1] + 1e-8, target_r, dtype=np.float64)
    new_c = np.arange(0.0, old_c[-1] + 1e-8, target_c, dtype=np.float64)
    new_z = np.arange(old_z[0], old_z[-1] + 1e-8, target_z, dtype=np.float64)

    interpolator = RegularGridInterpolator(
        points=(old_r, old_c, old_z),
        values=volume_hu,
        method="linear",   # в 3D это трилинейная интерполяция
        bounds_error=False,
        fill_value=float(fill_value),
    )

    new_volume = np.empty((len(new_r), len(new_c), len(new_z)), dtype=np.float32)

    rr, cc = np.meshgrid(new_r, new_c, indexing="ij")

    for k, z in enumerate(tqdm(new_z, desc="Resampling", leave=False)):
        pts = np.column_stack([
            rr.ravel(),
            cc.ravel(),
            np.full(rr.size, z, dtype=np.float64),
        ])
        new_slice = interpolator(pts).reshape(len(new_r), len(new_c))
        new_volume[:, :, k] = new_slice.astype(np.float32)

    return new_volume, new_r, new_c, new_z


def save_npz(volume_hu, out_path, spacing_rcz, z_positions):
    np.savez_compressed(
        out_path,
        volume=volume_hu.astype(np.float32),
        spacing=np.array(spacing_rcz, dtype=np.float32),
        z_positions=np.asarray(z_positions, dtype=np.float32),
        units="HU",
    )
def process_patient(patient_dir, output_dir,
                    target_spacing=(0.5, 0.5, 0.625),
                    fill_value=-1024.0):
    patient_dir = Path(patient_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    patient_id = patient_dir.name

    (
        volume_hu,
        row_spacing,
        col_spacing,
        z_positions,
        ipp0,
        row_dir,
        col_dir,
        slice_dir,
    ) = load_dicom_series(patient_dir)

    new_volume, new_r, new_c, new_z = resample_trilinear(
        volume_hu=volume_hu,
        row_spacing=row_spacing,
        col_spacing=col_spacing,
        z_positions=z_positions,
        target_spacing=target_spacing,
        fill_value=fill_value,
    )

    npz_path = output_dir / f"{patient_id}.npz"

    save_npz(
        volume_hu=new_volume,
        out_path=npz_path,
        spacing_rcz=target_spacing,
        z_positions=new_z,
    )
    return {
        "patient_id": patient_id,
        "npz_path": str(npz_path),
        "original_shape": volume_hu.shape,
        "resampled_shape": new_volume.shape,
        "hu_min": float(new_volume.min()),
        "hu_max": float(new_volume.max()),
    }


def process_dataset(dataset_dir, output_dir,
                    target_spacing=(0.5, 0.5, 0.625),
                    fill_value=-1024.0):
    dataset_dir = Path(dataset_dir)
    output_dir = Path(output_dir)

    patient_dirs = sorted([p for p in dataset_dir.iterdir() if p.is_dir()])

    results = []
    errors = []

    for patient_dir in tqdm(patient_dirs, desc="Patients"):
        try:
            res = process_patient(
                patient_dir=patient_dir,
                output_dir=output_dir,
                target_spacing=target_spacing,
                fill_value=fill_value,
            )
            results.append(res)
        except Exception as e:
            errors.append((patient_dir.name, str(e)))

    return results, errors


# =========================
# ЗАПУСК
# =========================
DATASET_DIR = r"D:\MosMedData-CT-HEMORRHAGE-type VIII\New_resample"
OUTPUT_DIR = r"D:\MosMedData-CT-HEMORRHAGE-type VIII\New_resample_result"

results, errors = process_dataset(
    dataset_dir=DATASET_DIR,
    output_dir=OUTPUT_DIR,
    target_spacing=(0.5, 0.5, 0.625),   # row, col, z
    fill_value=-1024.0,
)

print("Готово")
print("Успешно:", len(results))
print("Ошибок:", len(errors))

if results:
    print("Пример результата:")
    print(results[0])

if errors:
    print("\nПервые ошибки:")
    for patient_id, err in errors[:20]:
        print(patient_id, "->", err)

Визуализация файла формата npz

In [ ]:
# Если нужно, сначала установите:
# !pip install matplotlib ipywidgets
# и при необходимости выполните:
# %matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from pathlib import Path

# =========================
# УКАЖИТЕ ПУТЬ К ВАШЕМУ NPZ
# =========================
npz_path = Path(r"D:\MosMedData-CT-HEMORRHAGE-type VIII\New_resample_result\1.2.643.5.1.13.13.12.2.77.8252.00001510120413110515051009061303.npz")

# =========================
# ЗАГРУЗКА ДАННЫХ
# =========================
data = np.load(npz_path, allow_pickle=True)
volume = data["volume"]   # ожидается shape (Rows, Cols, Slices)
spacing = data["spacing"] if "spacing" in data.files else None

print("Файл:", npz_path)
print("Ключи в npz:", data.files)
print("shape:", volume.shape)
print("dtype:", volume.dtype)
print("min/max:", float(volume.min()), float(volume.max()))
if spacing is not None:
    print("spacing:", spacing)

# =========================
# ФУНКЦИЯ ПРОСМОТРА
# =========================
def show_slice(
    z,
    window_preset="brain",
    vmin_manual=-1000.0,
    vmax_manual=1000.0
):
    if window_preset == "brain":
        vmin, vmax = 0, 80
    elif window_preset == "bone":
        vmin, vmax = 300, 2000
    elif window_preset == "wide":
        vmin, vmax = -1024, 3071
    elif window_preset == "manual":
        vmin, vmax = vmin_manual, vmax_manual
    else:
        vmin, vmax = 0, 80

    plt.figure(figsize=(8, 8))
    plt.imshow(volume[:, :, z], cmap="gray", vmin=vmin, vmax=vmax)
    plt.title(f"Slice {z+1}/{volume.shape[2]}   |   window: {window_preset}   |   HU [{vmin}, {vmax}]")
    plt.axis("off")
    plt.show()

# =========================
# ИНТЕРАКТИВНЫЙ ПРОСМОТР
# =========================
interact(
    show_slice,
    z=IntSlider(
        min=0,
        max=volume.shape[2] - 1,
        step=1,
        value=volume.shape[2] // 2,
        description="Slice"
    ),
    window_preset=Dropdown(
        options=["brain", "bone", "wide", "manual"],
        value="brain",
        description="Window"
    ),
    vmin_manual=FloatSlider(
        min=float(volume.min()),
        max=float(volume.max()),
        step=1.0,
        value=max(float(volume.min()), -1024.0),
        description="vmin"
    ),
    vmax_manual=FloatSlider(
        min=float(volume.min()),
        max=float(volume.max()),
        step=1.0,
        value=min(float(volume.max()), 3071.0),
        description="vmax"
    ),
);